# Causal and Statistical Inference

This notebook picks up where [Regression](../06_regression/regression.ipynb) left off. There, we learned to estimate a regression and read its coefficients. Here, we ask two questions that lesson mostly set aside. First, when can a correlation (or a regression coefficient) be read as a **causal effect**: does $x$ actually *cause* $y$? Second, what do the standard errors, t-statistics, p-values, and stars in a regression table actually tell us, and what do they *not* tell us? In other words, what does **statistical significance** mean?

> **A note on scope.** This is a conceptual, discussion-driven lesson rather than a coding or inference tutorial. Causal inference is a field in its own right (several of the resources at the end of the notebook are whole books), and the goal here is to build intuition and a healthy skepticism, not to master the techniques. Several sections link out to published charts, cartoons, and articles rather than reproducing them here. Follow the links as you go.

Before starting, work through the [Regression](../06_regression/regression.ipynb) notebook. This module assumes you can estimate a regression with `statsmodels` and read summary outputs.

## General housekeeping items

As in the previous notebooks, select your kernel before running anything. Click **Select Kernel** in the top right of this notebook, choose **Python Environments...**, and select the `.venv` environment you created in [Getting Started](../01_getting_started/course_intro.ipynb).

The imports below are the same ones we used in [Regression](../06_regression/regression.ipynb), with one addition: `binom` from `scipy`, which gives us exact probabilities for the coin-flip example later in the notebook. We also apply the usual seaborn theme.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from scipy.stats import binom

sns.set_theme(style="whitegrid")

## Correlation or causation?

You have probably heard some variation of the phrase **"correlation does not imply causation."** We repeated it ourselves in [Regression](../06_regression/regression.ipynb) when we looked at the Kentucky game residual. The phrase is true, but it is also a little unsatisfying: correlations are often the *only* evidence we have, and some of them really do reflect cause and effect. The question is how to tell the difference. This section looks at three ways a correlation can mislead.

### Some correlations are due to chance

In 2022, as the Philadelphia Phillies advanced through the baseball playoffs, the [newsletter *Morning Brew*](https://x.com/MorningBrew/status/1572729134395969536) joked that over the past 100 years the surest sign of an oncoming financial crisis has been a Philadelphia-based baseball team winning the World Series. Look at enough pairs of series and some of them will line up by accident. Tyler Vigen's [Spurious Correlations](https://www.tylervigen.com/spurious-correlations) site is built entirely on this idea; browse a few of its charts.

#### In-class discussion

Pick one chart from Spurious Correlations. Why is it so easy to find pairs of series that move together? What does that tell you about the thousands of relations you could test in a large company's data? Keep this question in mind; it returns at the end of the notebook when we discuss data dredging and p-hacking.

### Some correlations are confounded

Other correlations are real, but the story behind them is more complicated than "x causes y." The U.S. Bureau of Labor Statistics publishes a chart every year showing that workers with more education earn more and are unemployed less often: [Education pays](https://www.bls.gov/emp/chart-unemployment-earnings-education.htm). In 2025, median weekly earnings for workers with a bachelor's degree were about $1,578, versus $966 for workers with only a high school diploma.

#### In-class discussion

Does a degree *cause* higher earnings? Probably, at least in part. But think about what else differs, on average, between people who finish a degree and people who do not: family background, test scores, the local job market, and so on. These other factors may be **confounders**: they push on both education and earnings at the same time, so some of the gap in the chart may reflect them rather than the degree itself. One way to think about causality in this context is in the form of a hypothetical question: For the same person in the same circumstances, would pursuing a degree improve their earnings?

### Non-correlation does not imply no causation

The reverse mistake is just as easy to make. Two variables can be causally linked and yet show little or no correlation in the data. That should sound a little weird to you. Let's look at this [chart of M2 money supply growth and the inflation rate](https://www.longtermtrends.com/m2-money-supply-vs-inflation/) (set the zoom to 30 years). For most of the 2000s and 2010s the two series barely move together, and yet most economists would say the money supply matters for inflation. Then look at 2020 to 2022.

#### In-class discussion

Why might two causally related series show little correlation most of the time? Consider: effects that arrive with a long lag, other forces pushing in the opposite direction, and a policymaker (here, the Federal Reserve) deliberately adjusting one variable in response to the other.

## Causal questions

Consider some questions we might genuinely want answered:

- Is a new drug effective in improving health outcomes?
- Does schooling improve future earnings?
- Do masks prevent transmission of an airborne disease?
- Did the Sarbanes-Oxley Act improve financial reporting?
- Do investors respond to material weakness disclosures?
- Is Sam the bartender skimming?

Each of these is a **causal** question: it asks what would happen to an outcome if we changed something. Notice that none of them can be answered by a correlation alone.

#### In-class discussion

Pick two of the questions above. If you wanted to answer them convincingly, what information (or data) would you want? What comparison would you want to make? We will come back to the last two questions later in the notebook.

## Experiments and observational studies

Studies of cause and effect come in two broad types:

- **Experimental studies** are settings where the researcher (or someone or something else) *intervenes* to assign conditions or treatments. A clinical trial that randomly assigns patients to a drug or a placebo is the classic example.
- **Observational studies** are settings where the researcher *observes* but does not intervene. Most accounting and finance data is observational: nobody randomly assigned companies to adopt a new standard or to disclose a material weakness.

### The experimental ideal

When investigating a causal effect, it helps to imagine the experiment you would run if you could. In an experiment, **random assignment** decides who gets the treatment, so on average the treated and untreated groups are alike in every other way, measured or not. Random assignment is a way of **holding all else equal**, which is exactly what the confounders in the education example were preventing. That is why experiments are the "gold standard" in causal inference. We will look at a real one, the Moderna COVID-19 vaccine trial, later in the notebook.

Often, we cannot run experiments and are limited to drawing inferences from observational data. To hold all else equal in observational settings, we rely on statistical techniques that attempt to isolate the causal connection. The technique we focus on in this course is **regression**, but there are several others (instrumental variables, difference-in-differences, regression discontinuity, and more; see the resources at the end - most are applications of regression techniques). Every one of them rests on assumptions that cannot be fully tested with the data.

> We should always use caution in drawing causal inferences from observational data, particularly when there is good reason to believe our model or data is incomplete or misspecified.

## A causal model: sunscreen and drownings

Suppose you are investigating the relation between sunscreen sales and drownings in a coastal town. The regression you learned in the last module is the natural first step:

$$drownings = \beta_0 + \beta_1\, sunscreen\_sales + \varepsilon$$

Estimate it on weekly data and you will almost certainly find a positive and "significant" $\beta_1$: weeks with more sunscreen sales have more drownings. Nobody believes sunscreen causes drowning. The point of the example is to see *why* the regression misleads, and what it would take to fix it.

If you want to determine whether sunscreen sales *cause* drownings, you need to hold all else equal (economists say *ceteris paribus*). In particular, we want to hold confounding factors equal. By adding variables to a regression model, we can adjust for the linear effects of those variables. To the extent we have the "correct" model, the coefficient on sunscreen sales then approximates its effect "holding all else equal":

$$drownings = \beta_0 + \beta_1\, sunscreen\_sales + \beta_k\, controls + \varepsilon$$

The extra variables are called **controls** or **control variables**. Here the obvious one is the weather: hot, sunny weekends send people to the beach, where they both buy sunscreen and, tragically, sometimes drown.

### Drawing the causal model

It helps to draw the story. The function below draws a simple three-box causal diagram: an $X$ we are interested in, an outcome $Y$, and a third variable $Z$ that may affect both. We will reuse it several times in this notebook. The code was built with Claude Code; do not worry about the details, which are just matplotlib boxes and arrows.

In [ ]:
def draw_causal_diagram(x_label, y_label, z_label, xy_sign='?',
                        zx_sign='+', zy_sign='+', title=None):
    # Draw three boxes (X, Y, Z) and the arrows Z -> X, Z -> Y, and X -> Y.
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis('off')

    box_style = dict(boxstyle='round,pad=0.6', facecolor='#cfe2f3',
                     edgecolor='#3d85c6', linewidth=1.5)
    ax.text(2.5, 4.6, f'X = {x_label}', ha='center', va='center',
            fontsize=12, bbox=box_style)
    ax.text(7.5, 4.6, f'Y = {y_label}', ha='center', va='center',
            fontsize=12, bbox=box_style)
    ax.text(2.5, 1.2, f'Z = {z_label}', ha='center', va='center',
            fontsize=12, bbox=box_style)

    arrow = dict(arrowstyle='-|>', linewidth=2.5, color='#3d85c6',
                 mutation_scale=22)
    ax.annotate('', xy=(5.9, 4.6), xytext=(4.1, 4.6), arrowprops=arrow)
    ax.annotate('', xy=(2.5, 3.9), xytext=(2.5, 1.9), arrowprops=arrow)
    ax.annotate('', xy=(6.4, 3.9), xytext=(3.9, 1.7), arrowprops=arrow)

    sign_style = dict(fontsize=14, fontweight='bold', color='#1c4587',
                      va='center')
    ax.text(5.0, 5.0, xy_sign, ha='center', **sign_style)
    ax.text(2.2, 2.9, zx_sign, ha='right', **sign_style)
    ax.text(5.5, 2.4, zy_sign, ha='center', **sign_style)

    if title:
        ax.set_title(title, fontsize=13)
    plt.show()


draw_causal_diagram('Sunscreen sales', 'Drownings', 'Weather conditions',
                    title='Sunscreen and drownings: a confounded relation')

In this example, "Z" (weather conditions) is a **confounder**: when we look at the relation between $X$ and $Y$, there is some other factor that determines both. If uncontrolled, a confounder makes $X$ and $Y$ appear related when they are not (or hides a relation that is really there, or changes its size). **Uncontrolled confounders are very bad for causal inference.**

You have already seen a confounder at work without naming it. In [Regression](../06_regression/regression.ipynb), the Intel coefficient in the CPU regression fell substantially once performance entered the model, because the Intel chips in that sample tended to be higher performance chips. Performance was confounding the brand effect. Adding it as a control is exactly the fix the second equation above describes, and we revisit that regression below.

Two cautions before moving on. First, a control variable only helps if you *have* it: the weather is easy to measure, but many confounders (a worker's ability, a company's culture) are not. Second, adding more variables to our model is not automatically better. Including a variable that is itself caused by $X$ and influences $Y$ can remove part of the causal effect we are trying to estimate. We will see an example of this in the bartender exercise. Which variables belong in the model is a question about the causal story, not a question the data can answer on its own. That's where you come in as a data scientist who understands the setting!

## Discussion examples

Three short items for discussion. For each one, identify $X$, $Y$, and the alternative story.

1. **xkcd 552, [Correlation](https://xkcd.com/552/).** "I used to think correlation implied causation. Then I took a statistics class. Now I don't." "Sounds like the class helped." "Well, maybe." Why is the punchline funny, and what would it take to know whether the class *did* help?

2. **Timo Elliott's cartoon, [Correlation vs Causation](https://timoelliott.com/blog/2023/11/correlation-vs-causation-cartoon.html).** A professor announces that "every single person who confuses correlation and causation ends up dying." The statement is true. What is wrong with the implied conclusion?

3. **Nate Silver, on X (July 5, 2024):** [Enron Troubles](https://x.com/NateSilver538/status/1809430521069597014) Draw the causal diagram for this one. Which direction do the arrows run?

## Exercise: Sam the bartender

You are working as a forensic data scientist at an accounting firm. One of your clients, the owner of a local restaurant, is concerned that Sam, a bartender, has been "skimming" from drink sales to customers. The basis for the owner's concern is that the accounting system clearly shows that sales and cash collections are consistently lower when Sam is working at the bar. The owner is considering firing Sam but asks you to look into it first.

You are not convinced by the validity of the owner's conclusion about Sam's culpability. **Come up with a plausible alternative explanation that might explain the seemingly suspicious pattern. Discuss what data and analyses you might use to confirm or disconfirm this explanation.**

Before reading further, discuss your explanation in class and sketch it on the diagram below. What goes in the $Z$ box?

In [ ]:
draw_causal_diagram("Sam's shifts", 'Bar sales', '?',
                    xy_sign='?', zx_sign='?', zy_sign='?',
                    title='Sam the bartender: what is Z?')

### A look at the data

Discussion done? Let's look at the numbers. The owner hands over a year of daily records: the date, the day of the week, who worked the bar, the hours the bar was open, the number of drinks made (computed from nightly inventory counts: the bar weighs its bottles at the end of every shift, so this number does not come from the register), and the sales rung up at the register. The file is `restaurant_sales.csv` in this folder (it is simulated for this course, which will matter at the end of the exercise):

In [ ]:
restaurant_sales = pd.read_csv('restaurant_sales.csv')

restaurant_sales.head(10)

Start where the owner started: average sales by bartender. We also create an indicator variable, `sam_working`, that equals 1 on the days Sam worked, the same way we built `is_intel` in [Regression](../06_regression/regression.ipynb), and regress sales on it. Recall from that module that a regression on a single indicator variable just compares two group means:

In [ ]:
restaurant_sales['sam_working'] = (
    restaurant_sales['bartender'] == 'Sam'
).astype(int)

print(restaurant_sales.groupby('bartender')['bar_sales']
      .agg(['mean', 'count']).round(0))

sam_reg_1 = smf.ols("bar_sales ~ sam_working", data=restaurant_sales).fit()

print(sam_reg_1.summary().tables[1])

The owner's instinct looks right: bar sales average about $340 less on the days Sam works, the coefficient has a large t-statistic, and the p-value is essentially zero. In the language of the next section, the "Sam effect" is highly statistically significant. Case closed?

Not so fast. This regression compares Sam's days with everyone else's days, and it treats every day as interchangeable. Is that true of a restaurant? Let's look at *which* days each bartender works, and how sales vary across the week:

In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday',
                 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.countplot(data=restaurant_sales, x='weekday', hue='bartender',
              order=weekday_order, hue_order=['Alex', 'Jordan', 'Sam'],
              ax=axes[0])
sns.move_legend(axes[0], 'upper center', ncol=3, title='Bartender')
axes[0].set_title('Shifts Worked by Bartender and Day of the Week')
axes[0].set_xlabel('Day of the Week')
axes[0].set_ylabel('Number of Shifts')
axes[0].set_ylim(0, 60)

sns.barplot(data=restaurant_sales, x='weekday', y='bar_sales',
            order=weekday_order, errorbar=None, color='steelblue',
            ax=axes[1])
axes[1].set_title('Average Bar Sales by Day of the Week')
axes[1].set_xlabel('Day of the Week')
axes[1].set_ylabel('Average Bar Sales ($)')

for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

There is one alternative explanation. Sam works mostly Sunday through Wednesday, the slow nights, while Alex and Jordan split the Friday and Saturday shifts, when the bar is busier and stays open an hour later. The day of the week is a confounder: it determines who is behind the bar *and* how much the bar sells. Comparing Sam's Tuesdays with Jordan's Saturdays tells us nothing about Sam.

The fix is the one from the sunscreen example: add the day of the week to the model as a control. With the `C(weekday)` indicators in the regression, the coefficient on `sam_working` now compares Sam with the other bartenders *on the same day of the week*, which is *closer to* holding all else equal. We collect both regressions in one table, as we did in [Regression](../06_regression/regression.ipynb):

In [ ]:
sam_reg_2 = smf.ols("bar_sales ~ sam_working + C(weekday)",
                    data=restaurant_sales).fit()

results_table = summary_col([sam_reg_1, sam_reg_2],
                            stars=True,
                            model_names=['(1)', '(2)'],
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

Compare the two columns. In column (1), Sam's days bring in about $340 less, with three stars. In column (2), once the day of the week is held constant, the gap shrinks to about $100, but it does not vanish: still three stars. The weekday coefficients tell the rest of the story (each is measured relative to Friday, the base category), and the r-squared jumps because the day of the week explains a large share of the variation in sales.

So the schedule explains most of the owner's pattern, but not all of it. Is the remaining $100 a night the skimming? Before answering, think about what skimming would actually look like in the data. A bartender who pockets cash makes drinks that never get rung up. The right place to look is not total sales but sales per drink made.

One caution matters here. The drink count has to come from somewhere the bartender cannot edit. The register only knows about drinks that were entered, so a count from the point-of-sale system would miss exactly the drinks a skimmer serves. But every drink, recorded or not, leaves the bottle. For illustration, let's say this bar weighs its bottles at the end of each shift (or otherwise counts inventory), the drinks made can be computed from inventory used, independent of the register. That gives us two numbers per bartender: how many drinks they make (which we calculate on per-hour basis) and how many dollars reach the register per drink made:

In [ ]:
restaurant_sales['drinks_per_hour'] = (
    restaurant_sales['drinks_made'] / restaurant_sales['shift_hours']
)
restaurant_sales['sales_per_drink'] = (
    restaurant_sales['bar_sales'] / restaurant_sales['drinks_made']
)

print(restaurant_sales
      .groupby('bartender')[['drinks_per_hour', 'sales_per_drink']]
      .mean()
      .round(2))

Two things stand out. Sam makes about 10% fewer drinks per hour than Alex and Jordan. Part of that is the slow nights he works, and part is that he is newer and less skilled behind the bar: slower with a crowd, and less likely to sell the second round. But every drink Sam makes brings about $9.00 to the register, the same as everyone else. That is the opposite of what skimming looks like. The remaining $100 is a *skill* story, not a fraud story: fewer drinks made, every one of them rung up.

We can confirm this with one more control. Adding the number of drinks made to the regression asks a sharper question: for a given number of drinks made, do Sam's nights bring in less money?

In [ ]:
sam_reg_3 = smf.ols("bar_sales ~ sam_working + C(weekday) + drinks_made",
                    data=restaurant_sales).fit()

results_table = summary_col([sam_reg_1, sam_reg_2, sam_reg_3],
                            stars=True,
                            model_names=['(1)', '(2)', '(3)'],
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

Column (3) answers it: with drinks made held constant, the Sam coefficient is essentially zero with no stars. Notice what each column asks. Column (1) asks whether sales are lower on Sam's nights (yes). Column (2) asks whether they are lower on Sam's nights *compared with the same day of the week* (yes, by about $100). Column (3) asks whether money goes missing between the bottle and the register on Sam's nights (no). Only the last one is the skimming question.

Because we simulated this data, we know the truth: on a comparable night Sam makes about 5% fewer drinks than his colleagues, and he rings up every one of them. The owner's pattern was created by the schedule and by Sam's skill, not by theft. Here is the diagram with the $Z$ box filled in; the arrow from $X$ to $Y$ is real, but it runs through Sam's skill, not his pocket:

In [ ]:
draw_causal_diagram("Sam's shifts", 'Bar sales', 'Day of the week',
                    xy_sign='skill only', zx_sign='Sam works\nslow days',
                    zy_sign='+',
                    title='Sam the bartender: the confounder revealed')

#### In-class discussion

- What did each control variable actually do? Explain in plain language why column (2) is a fairer comparison than column (1), and why column (3) is the one that tests for skimming.
- Column (3) is not a mechanical "add more variables" step. Drinks made sits *between* Sam and sales (his skill affects it), so adding it changes the question from "does Sam lower sales?" to "does money go missing per drink?" When would adding a variable like this be a mistake? (Recall the lifeguards from the sunscreen example.)
- Suppose the bar did not weigh its bottles and the only drink count came from the register. Why would that count be useless for testing skimming? What else could test it directly: cash drawer counts against register totals, test purchases, cameras?
- Does this prove Sam is honest? What could the inventory count miss (over-pouring, comped drinks, spillage)? What other confounders might remain (holidays, football Saturdays, the nights a band plays), and what additional data would you ask the owner for?
- In the real world you would not know the truth. How confident would you be telling the owner not to fire Sam, and what would you say about Sam's skill?

## A second causal model: bad news and stock prices

In early November 2019, the *Wall Street Journal* reported that federal investigators from the SEC and the Department of Justice had been examining, since 2017, whether the company shifted sales between quarters to make its results look better. Under Armour confirmed the investigations that evening and said it believed its accounting was appropriate. The next morning, before the market opened, the company released its third-quarter results and cut its full-year revenue forecast. 

Sources: [Fortune](https://fortune.com/2019/11/04/under-armour-shares-investigation-sec-doj), [Audit Analytics](https://blog.auditanalytics.com/under-armour-under-fire-for-undisclosed-investigations/).

The file `under_armour_2019.pkl` in this folder holds daily closing values for Under Armour's Class A stock (UAA) and the S&P 500 index for the ten weeks around the announcement, in the same layout as the market data from [Regression](../06_regression/regression.ipynb):

In [ ]:
under_armour = pd.read_pickle('under_armour_2019.pkl')

under_armour.head()

Stock prices and index levels are on different scales, so to plot them together we index both series to 100 on the first day of the window. Dividing each column by its first value and multiplying by 100 does this in one line. We then reshape to long format with `melt()` (as in [Working with Data](../04_working_data)) so that seaborn can draw one line per series, and mark November 4 with a dashed line:

In [ ]:
indexed = under_armour.set_index('date')
indexed = indexed / indexed.iloc[0] * 100

indexed_long = (
    indexed
    .reset_index()
    .melt(id_vars='date', var_name='series', value_name='indexed_value')
)
indexed_long['series'] = indexed_long['series'].map(
    {'sp500': 'S&P 500', 'uaa': 'Under Armour (UAA)'}
)

sns.lineplot(data=indexed_long, x='date', y='indexed_value', hue='series')

plt.axvline(pd.Timestamp('2019-11-04'), color='red', linestyle='--',
            label='Nov 4, 2019')
plt.title('Under Armour vs. the S&P 500, Fall 2019 (Sept 30 = 100)')
plt.xlabel('Date')
plt.ylabel('Indexed Value')
plt.legend()
plt.xticks(rotation=45)

plt.show()

The drop is hard to miss. Let's put a number on it with daily returns, computed with `pct_change()` as in the beta example, and compare Under Armour's return on November 4 with the market's:

In [ ]:
returns = under_armour.set_index('date').pct_change().dropna()

nov_4 = returns.loc['2019-11-04']

print(f"Under Armour return on Nov 4, 2019: {nov_4['uaa']:.1%}")
print(f"S&P 500 return on Nov 4, 2019:      {nov_4['sp500']:.1%}")
print(f"Market-adjusted return:             {nov_4['uaa'] - nov_4['sp500']:.1%}")

Under Armour lost about 19% of its value on a day the market rose. Subtracting the market's return (a version of what researchers call an **event study**) leaves a market-adjusted drop of about 19 percentage points. What exactly drove that drop in market value?

Three pieces of news reached investors in the same 24 hours: the investigation, the quarterly results, and the lower revenue forecast. Each one could move the stock on its own. The 19% drop is the *combined* reaction, and nothing in one day of one company's prices can split it into pieces. The other news confounds (or contaminates) the effect we want: it arrived together with the accounting news, and it affects the return.

This is not a quirk of Under Armour. Bad accounting news rarely travels alone. The simultaneous-news problem is one of the main reasons that researchers studying disclosure effects need many companies, careful comparison groups, and a good deal of judgment.

#### In-class discussion

- Suppose another apparel company had cut its revenue forecast by the same amount that week with no investigation. How would its stock reaction help you? What would you still not know?
- What data would you want in order to estimate the market's reaction to accounting investigations in general, rather than Under Armour's in particular?
- Under Armour had known about the investigation since 2017 and had not disclosed it. Does that change how you interpret the 19% drop?

## Our world is complex and so is our data

Step back from the two examples. Every causal question we have looked at came with a $Z$, and in most real settings there are many of them. The real world is complex, and modeling it honestly requires complex models: many control variables, careful thought about which ones belong, and an acceptance that some confounders cannot be measured at all.

Nonetheless, the better we design our models, the more meaningful our inferences. This is an area of continuous improvement, both for individual analysts and for the field. The takeaway is not "nothing can be known" but "know what your model holds constant, and be honest about what it does not."

## Overview of statistical significance

So far we have asked whether a relation is *causal*. A second, separate question is whether it is **statistically significant**. The two are often confused, so it is worth being precise.

What does it mean when a scientist, an auditor, or a news article says a result is "statistically significant"? Does it mean the result is:

- Important?
- Causal?
- Valid?

Google the word *significance* and the top results give the everyday meaning: importance, consequence. Be careful: that is **not** what the statistical term means. Now Google *statistical significance*. The [Wikipedia article](https://en.wikipedia.org/wiki/Statistical_significance) opens with: "a result has statistical significance when a result at least as extreme would be very infrequent if the null hypothesis were true." Much better, but kinda confusing... what exactly does that mean?

### Significance in a regression table

Every regression we estimated in [Regression](../06_regression/regression.ipynb) reported, for each coefficient, a **standard error**, a **t-statistic**, and a **p-value** (`P>|t|`). We postponed them to this module. Here is how they fit together:

- The **standard error** measures the precision of the coefficient estimate: how much the estimate would bounce around if we drew a different sample from the same population. (Recall from [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) how sample means scatter around the population mean; a coefficient is a sample statistic too.)
- The **t-statistic** is the coefficient divided by its standard error: how many standard errors the estimate sits from zero.
- The **p-value** converts the t-statistic into a probability: the probability of getting a t-statistic at least this far from zero if the true coefficient were zero.

Statistical significance is then determined by comparing the p-value with a cutoff, called **alpha** ($\alpha$). Typical cutoffs are 0.10, 0.05, and 0.01, and the `summary_col` tables we have been printing mark them with stars:

| Stars | Meaning |
|-------|---------|
| `*`   | p < 0.10 |
| `**`  | p < 0.05 |
| `***` | p < 0.01 |

The 0.05 cutoff is the most common convention, but it is only a convention. There is nothing magical about 0.049 that 0.051 lacks.

### Interpreting p-values

Read the definition slowly, because every word matters. A p-value is **the probability that we would observe an effect as large or larger than the one we estimated if the true effect were zero** (that is, if the null hypothesis were true and the estimate were just noise or randomness).

This is NOT the same as:

- The probability that the effect is true (or real).
- The probability that the effect is due to chance.
- The probability that the effect is economically meaningful.

The p-value starts from the assumption that there is no effect and asks how surprising the finding is. It doesn't tell you the probability that there *is* an effect; that would require knowing how common real effects are in the first place. The coin flips below make the distinction concrete.

## Illustration with coin flips

In [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) we imagined 1,000 students each flipping a fair coin 10 times, and one of the exercises asked what you would conclude if you were the student who flipped 9 heads. That exercise was secretly about the concepts underlying p-values. Let's redo the simulation with 10,000 flippers, written more compactly: `np.random.randint` draws all of the flips at once as a 10,000-by-10 table of zeros and ones, and summing each row counts the heads. Then we plot the share of flippers at each number of heads:

In [ ]:
np.random.seed(42)

n_flippers = 10000
n_flips = 10

flips = np.random.randint(0, 2, size=(n_flippers, n_flips))
heads = pd.DataFrame({'heads': flips.sum(axis=1)})

heads_share = (
    heads
    .groupby('heads')
    .size()
    .div(n_flippers)
    .reset_index(name='share')
)

sns.barplot(data=heads_share, x='heads', y='share', color='dimgray')

plt.title(f'Share of {n_flippers:,} Fair-Coin Flippers by Number of Heads')
plt.xlabel(f'Heads in {n_flips} Flips')
plt.ylabel('Share of Flippers')

plt.show()

This is the same bell-shaped pattern we saw in the sampling module. Now suppose you are a flipper who got **9 heads** in 10 flips, and you suspect your coin is loaded. The null hypothesis is "the coin is fair." The p-value is the probability that a *fair* coin produces a result at least as extreme as yours. "At least as extreme" includes 9 or 10 heads and, because a loaded coin could favor tails just as well, 0 or 1 heads too (this is a *two-sided* test, which is what a regression table reports). We can read the p-value straight off the simulation, and `binom` from `scipy` gives the exact probability for comparison:

In [ ]:
extreme = (heads['heads'] >= 9) | (heads['heads'] <= 1)

p_simulated = extreme.mean()
p_exact = binom.cdf(1, n_flips, 0.5) + binom.sf(8, n_flips, 0.5)

print(f"Simulated p-value (share of fair coins at least this extreme): "
      f"{p_simulated:.3f}")
print(f"Exact p-value from the binomial distribution:                  "
      f"{p_exact:.3f}")

About 2% of fair coins produce a result as lopsided as yours, so the p-value is roughly 0.02: "significant at the 5% level," two stars in a regression table. Now re-read the list of things a p-value is not. The 2% is **not** the probability that your coin is fair. If every coin on campus is fair, then *every* flipper with 9 heads (about 100 of our 10,000) is holding a fair coin, despite a significant result. The p-value only tells you that a fair coin would rarely do this. Whether a loaded coin is a plausible alternative is a separate question, and one the p-value doesn't directly answer.

**Two caveats about this example.** First, it is cleaner than real data. We know the null hypothesis exactly (a fair coin lands heads half the time) and the flips are independent (one flip doesn't affect another). In a regression, the standard errors that produce the p-value are themselves estimated from the data, and they depend on the model being right: the correct variables, independent observations, and so on. Second, we picked the flipper to examine *after* seeing the results. A p-value assumes you decided what to test before you looked. 

#### In-class discussion

Change `n_flips` to 100 and rerun the simulation cell. What happens to the spread of the distribution (look at the x-axis)? With 100 flips, 90 heads would be astronomically rare, but 55 heads would not be surprising at all. Why does a bigger sample make small departures from 50% easier to detect? This is exactly why standard errors shrink, and stars multiply, as the number of observations in a regression grows.

## A real experiment: the Moderna COVID-19 vaccine trial

Here is what the experimental ideal looks like in practice, with p-values attached. In Moderna's phase 3 trial, 30,420 participants were randomized one-to-one to receive either the vaccine or a placebo and then followed for symptomatic COVID-19. The original results page is no longer online, but it is preserved on the [Wayback Machine](http://web.archive.org/web/20220928100904/https://eua.modernatx.com/covid19vaccine-eua/providers/clinical-trial-data-primary-series) (snapshot from September 28, 2022). The key numbers:

| Group | Participants | COVID-19 cases |
|---------|-------------:|---------------:|
| Vaccine | 14,134 | 11 |
| Placebo | 14,073 | 185 |

Vaccine efficacy: 94.1% (95% confidence interval, 89.3% to 96.8%), p < 0.001.

Note: The statistical procedures used to generate these statistics are a little different than the basic ones we use in this class, but the underlying idea and interpretations apply. See link above for discussion.

#### In-class discussion

- Why can this study support a *causal* claim ("the vaccine prevents COVID-19") when the education chart could not? What did random assignment do to all of the possible $Z$ variables?
- Put the p-value into words using the definition above. What is the null hypothesis, and what does "p < 0.001" say about the data under that hypothesis?

## CPU regression: economic significance

Statistical significance is not the same as **economic significance**: whether an effect is large enough to matter. The best way to see the difference is a large sample. Let's return to the CPU regressions from [Regression](../06_regression/regression.ipynb): the natural log of price regressed on the natural log of the PassMark performance score, on brand, and on both. The data file is a copy of the one in that module:

In [ ]:
cpu_data = pd.read_csv('cpu_data.csv')

cpu_data['log_price'] = np.log(cpu_data['price_usd'])
cpu_data['log_performance'] = np.log(cpu_data['cpu_mark'])

cpu_reg_1 = smf.ols("log_price ~ log_performance", data=cpu_data).fit()
cpu_reg_2 = smf.ols("log_price ~ C(brand)", data=cpu_data).fit()
cpu_reg_3 = smf.ols("log_price ~ log_performance + C(brand)",
                    data=cpu_data).fit()

results_table = summary_col([cpu_reg_1, cpu_reg_2, cpu_reg_3],
                            stars=True,
                            model_names=['(1)', '(2)', '(3)'],
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

#### In-class discussion

- **Can you interpret the coefficients?** Note that performance and price are both in natural log form, so the performance coefficient is an elasticity (see the log section of [Regression](../06_regression/regression.ipynb)). The brand coefficient is a difference in log price; converting it with $e^{0.229} - 1$ (26%) gives the Intel premium as a percentage.
- **Why does the Intel coefficient fall from column (2) to column (3)?** Answer in the language of this notebook: what is $X$, what is $Y$, and what is $Z$?
- **What about statistical significance?** Every coefficient has three stars. Does that make each effect important?
- **R-squared?** Performance explains about half of the variation in log price. What might explain the other half?

## Words of caution with p-values

To summarize what we have seen:

- **P-values do not tell us that an effect is "true" or "important."** They tell us how surprising the data would be if there were no effect.
- **The practical use of p-values is susceptible to false positives**, especially when a researcher sorts through many variables, samples, and model specifications looking for p-values below a threshold. This practice goes by the names **p-hacking** and **data dredging**. (Andrew Gelman, whose blog is linked at the end, calls the subtler, unintentional version "the garden of forking paths": every small analytical choice is a fork, and an analyst who takes the forks that lead to significance will find it.)

The classic illustration is xkcd 882, [Significant](https://xkcd.com/882/): scientists test whether jelly beans cause acne, find nothing, then test 20 colors separately and find that green jelly beans are linked to acne at p < 0.05. The newspaper headline writes itself.

### Simulating the jelly bean study

Let's run the comic. Below, we create 1,000 observations in which the outcome (`acne`) and 20 "jelly bean color" variables are all pure random noise; by construction, nothing causes anything. Then we regress acne on each color, one at a time, exactly as the scientists in the comic did, and collect the p-values. At a 5% cutoff, we should expect about 1 false positive in 20 tries:

In [ ]:
np.random.seed(7)

n_obs = 1000
colors = ['purple', 'brown', 'pink', 'blue', 'teal', 'salmon', 'red',
          'turquoise', 'magenta', 'yellow', 'grey', 'tan', 'cyan', 'green',
          'mauve', 'beige', 'lilac', 'black', 'peach', 'orange']

jelly_beans = pd.DataFrame(np.random.normal(size=(n_obs, len(colors) + 1)),
                           columns=['acne'] + colors)

jelly_results = []
for color in colors:
    reg = smf.ols(f"acne ~ {color}", data=jelly_beans).fit()
    jelly_results.append({'color': color,
                          'coef': reg.params[color],
                          't': reg.tvalues[color],
                          'p_value': reg.pvalues[color]})

jelly_results = pd.DataFrame(jelly_results)
jelly_results['stars'] = pd.cut(jelly_results['p_value'],
                                bins=[0, 0.01, 0.05, 0.10, 1],
                                labels=['***', '**', '*', ''])

print(jelly_results.round(3).to_string(index=False))
print(f"\nSignificant at the 5% level: "
      f"{(jelly_results['p_value'] < 0.05).sum()} of {len(colors)} colors")

Scan the stars column. Even though every variable is noise, the search turned up a "significant" color or two, and had we reported only that regression, it would look exactly like a real finding. Change the seed in the first line and rerun the cell: sometimes you will find no false positives, sometimes two or three. Over many runs, the average is one in twenty, which is precisely what a 5% cutoff promises when the null hypothesis is true.

The problem is not the p-value; it is the *search*. Twenty independent tests at 5% give a roughly 64% chance of at least one false positive ($1 - 0.95^{20}$). The same arithmetic applies to an analyst who tries twenty model specifications, twenty subsamples, or twenty outcome variables and reports the one that "worked." The remedies are mostly about  and transparency: decide what you will test before you look, report everything you tried, treat a surprising result from a big search as a hypothesis to test on *new* data rather than a conclusion, and remember the spurious correlations from the start of the notebook.

#### In-class discussion

Return to the Spurious Correlations site. In light of this simulation, explain why it is so easy to find series that line up. Then think about an audit: if you screen thousands of transactions, vendors, or employees for "unusual" patterns at a 5% threshold, how many false alarms should you expect, and what does that imply for how you follow up on them?

## Overall takeaways

- **Regressions can be used to predict outcomes or to determine how variables are related to one another.** Prediction does not require a causal story per se; explaining *why* does.
- **Uncovering causal effects requires assumptions and a firm understanding of the setting and the data.** Ask what the experiment would look like, draw the diagram, name the confounders, and be upfront about the ones you cannot measure.
- **Statistical significance is not the same as economic significance!** A p-value says how surprising the data would be if there were no effect. It does not say the effect is real, large, causal, or important. Also, with a sufficiently large sample, even very small effects can become statistically significant.

## Other resources and acknowledgements

Credit to the following resources for solidifying my understanding of these issues and improving my ability to communicate causal issues in understandable terms:

- Angrist and Pischke, [*Mastering 'Metrics: The Path from Cause to Effect*](https://www.masteringmetrics.com/), an accessible book that builds causal inference around the experimental ideal.
- Andrew Gelman's blog, [*Statistical Modeling, Causal Inference, and Social Science*](https://statmodeling.stat.columbia.edu/), a running commentary on good and bad statistical practice.

I also encourage Scott Cunningham's [*Causal Inference: The Mixtape*](https://mixtape.scunning.com/), which is free online and includes worked examples in Python and R.

## Wrapping up

In this notebook you learned to tell a correlation from a causal effect, to draw a causal diagram and name its confounders, to use control variables in a regression to hold all else equal (and to recognize when they cannot), and to read the standard errors, t-statistics, p-values, and stars in a regression table for what they are: a measure of how surprising the data would be under a null hypothesis, and nothing more. Regression is a tool for both explanation and prediction.